# Demo: section 11, calculate probability distributions from Clinton-Gore data

20260726

Author: Kyoko Kusano

This library reproduces the calculations in Section 11 of Ozawa and Khrennikov(2021), “Modeling combination of question order effect, response replicability effect, and QQ-equality with quantum instruments.”

In [2]:
from qinst_ozawa import (
    CLINTON_GORE,
    BLACK_WHITE,
    ROSE_JACKSON,
    SequentialProbabilities,
    fit_independent_model,
    qq_residual,
    qqe_renormalize,
    reconstruct_jointprobdists,
    BeliefDistribution,
    PersonalityDistribution,
    IndependentModelParameters,
)

In [32]:
# read Clinton-Gore poll data: joint probablities
original = SequentialProbabilities.from_mapping(CLINTON_GORE)
CLINTON_GORE

# Or declare like this:
# original = SequentialProbabilities(
#     ay_by= 0.4899,
#     ay_bn= 0.0447,
#     an_by= 0.1767,
#     an_bn= 0.2887,
#     by_ay= 0.5625,
#     by_an= 0.1991,
#     bn_ay= 0.0255,
#     bn_an= 0.2129,
# )

{'AyBy': 0.4899,
 'AyBn': 0.0447,
 'AnBy': 0.1767,
 'AnBn': 0.2887,
 'ByAy': 0.5625,
 'ByAn': 0.1991,
 'BnAy': 0.0255,
 'BnAn': 0.2129}

In [38]:
CLINTON_GORE_inv: dict[str, float] = {
    "ByAy": 0.4899,
    "ByAn": 0.0447,
    "BnAy": 0.1767,
    "BnAn": 0.2887,
    "AyBy": 0.5625,
    "AyBn": 0.1991,
    "AnBy": 0.0255,
    "AnBn": 0.2129,
}

inv = SequentialProbabilities.from_mapping(CLINTON_GORE_inv)

p_ay = inv.ay_by + inv.ay_bn
p_an = inv.an_by + inv.an_bn
p_by = inv.by_ay + inv.by_an
p_bn = inv.bn_ay + inv.bn_an

print(f"ay: {p_ay: .4f}, by: {p_by: .4f},  bn: {p_bn: .4f}, an: {p_an: .4f}")

# CLINTON_GORE: inverse a/b

# Thm. 10.1 condition (iii) 1-q(1) vs q(2) -> q(0) < 0
print("condition iii")
print((inv.ay_bn - inv.by_an) / (p_ay - p_by))
print((inv.ay_bn - inv.bn_ay) / (p_ay - p_bn))


# Thm. 10.1 condition (iv): symmetry A-B
print("condition iv")
print(inv.by_ay / p_by)
print(inv.ay_by / p_ay)

print("condition v")
print(inv.bn_ay / p_bn)
print(inv.ay_bn / p_ay)

print("condition vi")
print(inv.an_by / p_an)
print(inv.by_an / p_by)

# Thm. 10.1 condition (vii): symmetry A-B
print("condition vii")
print(inv.an_bn / p_an)
print(inv.bn_an / p_bn)

ay:  0.7616, by:  0.5346,  bn:  0.4654, an:  0.2384
condition iii
0.6801762114537443
0.07562457798784605
condition iv
0.9163860830527498
0.7385766806722689
condition v
0.3796733992264718
0.26142331932773105
condition vi
0.10696308724832214
0.08361391694725027
condition vii
0.8930369127516778
0.6203266007735281


In [37]:
# CLINTON_GORE: not inverse a/b

p_ay = original.ay_by + original.ay_bn
p_an = original.an_by + original.an_bn
p_by = original.by_ay + original.by_an
p_bn = original.bn_ay + original.bn_an

print(f"ay: {p_ay: .4f}, by: {p_by: .4f},  bn: {p_bn: .4f}, an: {p_an: .4f}")

# Thm. 10.1 condition (iii) 1-q(1) vs q(2) -> q(0) < 0
print("condition iii")
print((original.ay_bn - original.by_an) / (p_ay - p_by))
print((original.ay_bn - original.bn_ay) / (p_ay - p_bn))


# Thm. 10.1 condition (iv): symmetry A-B
print("condition iv")
print(original.by_ay / p_by)
print(original.ay_by / p_ay)

print("condition v")
print(original.bn_ay / p_bn)
print(original.ay_bn / p_ay)

print("condition vi")
print(original.an_by / p_an)
print(original.by_an / p_by)

# Thm. 10.1 condition (vii): symmetry A-B
print("condition vii")
print(original.an_bn / p_an)
print(original.bn_an / p_bn)

condition iii
0.6801762114537443
0.06482106684672519
condition iv
0.7385766806722689
0.9163860830527498
condition v
0.10696308724832214
0.08361391694725027
condition vi
0.3796733992264718
0.26142331932773105
condition vii
0.6203266007735281
0.8930369127516778


In [24]:
# BLACK_WHITE: inverse y/n

BLACK_WHITE_inv: dict[str, float] = {
    "AnBn": 0.3987,
    "AnBy": 0.0174,
    "AyBn": 0.1612,
    "AyBy": 0.4227,
    "BnAn": 0.4012,
    "BnAy": 0.0597,
    "ByAn": 0.1379,
    "ByAy": 0.4012,
}

inv = SequentialProbabilities.from_mapping(BLACK_WHITE_inv)

print(inv)

ay: 0.4161, an: 0.5839000000000001, by: 0.4609, bn: 0.5391
SequentialProbabilities(ay_by=0.4227, ay_bn=0.1612, an_by=0.0174, an_bn=0.3987, by_ay=0.4012, by_an=0.1379, bn_ay=0.0597, bn_an=0.4012)


In [31]:
p_ay = inv.ay_by + inv.ay_bn
p_an = inv.an_by + inv.an_bn
p_by = inv.by_ay + inv.by_an
p_bn = inv.bn_ay + inv.bn_an

# Thm. 10.1 condition (iii) 1-q(1) vs q(2) -> q(0) < 0
print("condition iii")
print((inv.ay_bn - inv.by_an) / (p_ay - p_by))
print((inv.ay_bn - inv.bn_ay) / (p_ay - p_bn))


# Thm. 10.1 condition (iv)
print("condition iv")
print(inv.by_ay / p_by)
print(inv.ay_by / p_ay)

print("condition v")
print(inv.bn_ay / p_bn)
print(inv.ay_bn / p_ay)

print("condition vi")
print(inv.an_by / p_an)
print(inv.by_an / p_by)

print("condition vii")
print(inv.an_bn / p_an)
print(inv.bn_an / p_bn)

condition iii
0.5200892857142854
0.8252032520325197
condition iv
0.7442033017992951
0.7239253296797397
condition v
0.12952918203514863
0.2760746703202603
condition vi
0.0418168709444845
0.25579669820070483
condition vii
0.9581831290555154
0.8704708179648514


In [43]:
# ROSE_JACKSON

original = SequentialProbabilities.from_mapping(ROSE_JACKSON)
ROSE_JACKSON

p_ay = original.ay_by + original.ay_bn
p_an = original.an_by + original.an_bn
p_by = original.by_ay + original.by_an
p_bn = original.bn_ay + original.bn_an

print(f"ay: {p_ay:.4f}, by: {p_by:.4f}, bn: {p_bn:.4f}, an: {p_an:.4f}")

ay: 0.6620, by: 0.4827, bn: 0.5173, an: 0.3380


In [51]:
# Question A: still, Question B: inverse y/n

ROSE_JACKSON_inv: dict[str, float] = {
    "AyBn": 0.3379,
    "AyBy": 0.3241,
    "AnBn": 0.0178,
    "AnBy": 0.3202,
    "BnAy": 0.4156,
    "BnAn": 0.0671,
    "ByAy": 0.1234,
    "ByAn": 0.3939,
}

inv = SequentialProbabilities.from_mapping(ROSE_JACKSON_inv)

p_ay = inv.ay_by + inv.ay_bn
p_an = inv.an_by + inv.an_bn
p_by = inv.by_ay + inv.by_an
p_bn = inv.bn_ay + inv.bn_an

print(f"ay: {p_ay:.4f}, by: {p_by:.4f}, bn: {p_bn:.4f}, an: {p_an:.4f}")


# condition (i) # numerator of q(2)
print("condition i")
print(inv.ay_by)
print(inv.by_ay)

# condition (ii) # numerator of q(1)
print("condition ii")
print(inv.ay_bn)
print(inv.bn_ay)

# Thm. 10.1 condition (iii) 1-q(1) - q(2) < 0  -> q(0) < 0
print("condition iii")
print((inv.ay_bn - inv.by_an) / (p_ay - p_by))
print((inv.ay_bn - inv.bn_ay) / (p_ay - p_bn))


# Thm. 10.1 condition (iv)
print("condition iv")
print(inv.by_ay / p_by)
print(inv.ay_by / p_ay)

print("condition v")
print(inv.bn_ay / p_bn)
print(inv.ay_bn / p_ay)

print("condition vi")
print(inv.an_by / p_an)
print(inv.by_an / p_by)

print("condition vii")
print(inv.an_bn / p_an)
print(inv.bn_an / p_bn)

ay: 0.6620, by: 0.5173, bn: 0.4827, an: 0.3380
condition i
0.3241
0.1234
condition ii
0.3379
0.4156
condition iii
-0.38700760193503814
-0.43335192414947066
condition iv
0.23854629808621688
0.4895770392749245
condition v
0.8609902631033769
0.5104229607250755
condition vi
0.9473372781065089
0.761453701913783
condition vii
0.05266272189349113
0.13900973689662316


In [52]:
# calculate how much this data violates QQE

print(f"{qq_residual(inv)*100:.4f}%")

-15.1400%


In [53]:
# renormalize data to satisfy QQE

renormalized = qqe_renormalize(inv)

probability_fields = (
    ("AyBy", "ay_by"),
    ("AyBn", "ay_bn"),
    ("AnBy", "an_by"),
    ("AnBn", "an_bn"),
    ("ByAy", "by_ay"),
    ("ByAn", "by_an"),
    ("BnAy", "bn_ay"),
    ("BnAn", "bn_an"),
)

print("Observed → QQE-renormalized")
for paper_name, field_name in probability_fields:
    original_value = getattr(renormalized.original, field_name)
    normalized_value = getattr(renormalized.normalized, field_name)
    print(
        f"{paper_name} ({field_name}): "
        f"{original_value:.4f} → {normalized_value:.4f}"
    )

print(f"\nS1: {renormalized.s1:.4f}")
print(f"S2: {renormalized.s2:.4f}")
print(
    "QQ residual after renormalization: "
    f"{qq_residual(renormalized.normalized) * 100:.4f}%"
)

Observed → QQE-renormalized
AyBy (ay_by): 0.3241 → 0.2523
AyBn (ay_bn): 0.3379 → 0.3768
AnBy (an_by): 0.3202 → 0.3570
AnBn (an_bn): 0.0178 → 0.0139
ByAy (by_ay): 0.1234 → 0.1724
ByAn (by_an): 0.3939 → 0.3571
BnAy (bn_ay): 0.4156 → 0.3767
BnAn (bn_an): 0.0671 → 0.0938

S1: 0.2662
S2: 0.7338
QQ residual after renormalization: 0.0000%


In [54]:
# estimate the parameters of the model

params = fit_independent_model(renormalized.normalized)

print("Personality distribution q(gamma)")
print(f"q0 = q(gamma=0): {params.personality.q0:.4f}")
print(f"q1 = q(gamma=1): {params.personality.q1:.4f}")
print(f"q2 = q(gamma=2): {params.personality.q2:.4f}")

print("\nBelief distribution p(A, B)")
print(f"p11 = p(A=y, B=y): {params.belief.p11:.4f}")
print(f"p10 = p(A=y, B=n): {params.belief.p10:.4f}")
print(f"p01 = p(A=n, B=y): {params.belief.p01:.4f}")
print(f"p00 = p(A=n, B=n): {params.belief.p00:.4f}")

ValueError: The data are not representable by the independent-personality model: the inferred belief distribution is invalid (Belief probabilities must be between 0 and 1; received -1.435540757557361.).

In [7]:
# reconstruct the data from parameters

reconst = reconstruct_jointprobdists(params)

print("Observed → QQE-renormalized → Reconstructed")
for paper_name, field_name in probability_fields:
    original_value = getattr(original, field_name)
    normalized_value = getattr(renormalized.normalized, field_name)
    reconstructed_value = getattr(reconst, field_name)
    print(
        f"{paper_name} ({field_name}): "
        f"{original_value:.4f} → {normalized_value:.4f} → "
        f"{reconstructed_value:.4f}"
    )

largest_reconstruction_error = max(
    abs(
        getattr(reconst, field_name)
        - getattr(renormalized.normalized, field_name)
    )
    for _, field_name in probability_fields
)
print(
    "\nLargest difference between the QQE-renormalized and "
    f"reconstructed values: {largest_reconstruction_error:.4f}"
)

Observed → QQE-renormalized → Reconstructed
AyBy (ay_by): 0.4899 → 0.4889 → 0.4889
AyBn (ay_bn): 0.0447 → 0.0450 → 0.0450
AnBy (an_by): 0.1767 → 0.1780 → 0.1780
AnBn (an_bn): 0.2887 → 0.2881 → 0.2881
ByAy (by_ay): 0.5625 → 0.5637 → 0.5637
ByAn (by_an): 0.1991 → 0.1977 → 0.1977
BnAy (bn_ay): 0.0255 → 0.0253 → 0.0253
BnAn (bn_an): 0.2129 → 0.2133 → 0.2133

Largest difference between the QQE-renormalized and reconstructed values: 0.0000


In [8]:
# reconstruct joint probabilities from arbitary parameters

beliefs = BeliefDistribution(
    p00=0.2,
    p10=0.1,
    p01=0.4,
    p11=0.3,
)

personalities = PersonalityDistribution(
    q0=0.2,
    q1=0.4,
    q2=0.4,
)

params_a = IndependentModelParameters(
    personality = personalities,
    belief = beliefs,
)

reconstruct_jointprobdists(params_a)

SequentialProbabilities(ay_by=0.22000000000000003, ay_bn=0.18000000000000005, an_by=0.32000000000000006, an_bn=0.28, by_ay=0.33999999999999997, by_an=0.36, bn_ay=0.14, bn_an=0.16000000000000003)